# Comparación H0 vs H3 vs PPO -- escenario chico (2 barcos, escalón 1)

**Qué es esto, en una frase:** mismo tipo de comparación que `comparacion/dia_completo/` pero en el escenario MÁS SIMPLE del proyecto (`escalón 1`: 2 barcos, 06:00-09:00, ~0.8% de la población) -- más fácil de seguir en una presentación -- y agrega **H3** a la comparación que ya existía en `comparacion/notebooks/05_comparacion_agente_vs_base.ipynb` (que solo tenía PPO vs. H0/política base).

**Diferencia clave con `dia_completo/`:** aquí las tres políticas (H0, H3, PPO) corren sobre el MISMO entorno (`EntornoDemandaAleatoria`) con la MISMA fórmula de recompensa (Reward D, `agente.entrenamiento.recompensa_overrides`) -- exactamente el patrón ya establecido en `comparacion/notebooks/05`. Eso quiere decir que aquí el desglose de recompensa **SÍ es comparable entre políticas** (a diferencia de `dia_completo/`, donde H0/H3 corrían con la fórmula de producción). Aun así, el veredicto sigue liderando con las métricas operativas, no con el reward -- mismo criterio anti-Goodhart del resto del proyecto.

**Modelo usado:** `modelo_rl/prueba_rewards/output/modelos/D/` -- **NO** `modelo_rl/output/modelo_final/` (usado en una versión anterior de este notebook). Ambos son reward D, escalón 1, 150 000 timesteps, `semilla_entrenamiento=123` -- misma config nominal, pero son dos corridas de entrenamiento DISTINTAS (verificado por hash SHA-256, los pesos no son idénticos, la estocasticidad de PPO produjo dos políticas distintas). `prueba_rewards/D` es la que da resultados notablemente mejores (más cerca de H0) -- confirmado abajo, no solo de memoria.

**Animaciones: VIDEO (.mp4), no GIF.** Un GIF no se puede pausar de forma confiable insertado en PowerPoint -- un .mp4 sí (controles nativos de reproducción). Se generan con `imageio` + `imageio-ffmpeg` (instalados para esto), reusando `visualizacion.dibujar_frame` (sin modificar ese archivo salvo dos textos que se tradujeron a inglés -- "waiting:" y el título del mapa -- para que TODAS las animaciones del proyecto queden en inglés).

**Reproductor interactivo (paso a paso, panel de estado en inglés):** para cada política individual (mejor y peor semilla) se agrega un reproductor con slider + botones + panel de texto (estado de cada barco, colas, decisión tomada, recompensa del paso) -- para H3 el panel también muestra las reservas activas (a qué cola está comprometido cada barco). Solo funciona con un kernel de Jupyter VIVO (igual que `visualizacion.reproductor_interactivo`, del que se reusa el patrón) -- no se ve funcionando en este notebook ya guardado, hay que abrirlo y correr esa celda.

No se modificó el simulador, las heurísticas, el modelo entrenado, ninguna métrica existente, ni GitHub.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots
import imageio

BASE = Path.cwd().resolve().parent.parent.parent   # notebooks/ -> escalon_1_h0_h3_ppo/ -> comparacion/ -> raiz del repo
assert (BASE / "simulador").exists(), f"BASE mal calculado: {BASE}"

sys.path.insert(0, str(BASE / "demand" / "src"))
sys.path.insert(0, str(BASE / "simulador" / "src"))
sys.path.insert(0, str(BASE / "politica_base" / "src"))
sys.path.insert(0, str(BASE / "modelo_rl" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "comun" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h3_costo_global" / "src"))

import masas as demand_masas                            # noqa: E402
import llegadas as demand_llegadas                       # noqa: E402
from entrenamiento import EntornoDemandaAleatoria         # noqa: E402
from politica_base import asignar_flota                  # noqa: E402 -- H0
from politica_h3 import asignar_flota_h3                  # noqa: E402 -- H3 (nuevo aqui, no estaba en 05)
import metricas as met                                     # noqa: E402
import visualizacion as viz                                 # noqa: E402
from stable_baselines3 import PPO                            # noqa: E402
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize  # noqa: E402

OUT_DIR = Path.cwd().resolve().parent   # comparacion/escalon_1_h0_h3_ppo/
FIG_DIR = OUT_DIR / "figuras"
RES_DIR = OUT_DIR / "resultados"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

COLOR = {"H0": "#1f77b4", "H3": "#2ca02c", "PPO": "#e67e22"}
print("BASE =", BASE)
print("OUT_DIR =", OUT_DIR)

BASE = C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen
OUT_DIR = C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo


In [2]:
SIMULADOR_DIR = BASE / "simulador"
BB_DIR = BASE / "bergen-boats"
DEMAND_DIR = BASE / "demand"
MODELO_RL_DIR = BASE / "modelo_rl"

cfg_sim = yaml.safe_load(open(SIMULADOR_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]

nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

cfg_agente = cfg_sim["agente"]
cfg_entren = cfg_agente["entrenamiento"]
cfg_escalon = cfg_sim["escalones"][cfg_entren["escalon_base"]]   # escalon_1 -- 2 barcos
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60
SEMILLAS_EVAL = cfg_agente["evaluacion"]["semillas"]
capacidad = cfg_bb["flota"]["capacidad_pasajeros"]

# Config de recompensa D -- UNA sola fuente, usada para las TRES politicas
# (mismo patron que comparacion/notebooks/05, extendido a H3 aqui).
cfg_recompensa_eval = dict(cfg_sim["recompensa"])
cfg_recompensa_eval.update(cfg_entren.get("recompensa_overrides", {}))

RL_DIR = MODELO_RL_DIR / "prueba_rewards" / "output" / "modelos" / "D"
RL_DIR_VIEJO = MODELO_RL_DIR / "output" / "modelo_final"   # solo para el chequeo de hash de abajo -- no se usa para nada mas
print(f"Escenario: {cfg_entren['escalon_base']} -- {cfg_escalon['num_barcos']} barcos, "
      f"horas {cfg_escalon['horas']}, {cfg_escalon['porcentaje_poblacion_dia']*100:.1f}% poblacion")
print(f"Semillas de evaluacion: {SEMILLAS_EVAL}")
print(f"Config de recompensa D (compartida por H0, H3 y PPO): {cfg_recompensa_eval}")

# Transparencia: confirmar que RL_DIR (prueba_rewards/D) y modelo_final NO son
# el mismo archivo (misma config nominal -- reward D, 150k timesteps, semilla
# 123 -- pero corridas de entrenamiento distintas, confirmado por hash).
import hashlib
def _sha256(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()
_h_nuevo = _sha256(RL_DIR / "modelo_ppo.zip")
_h_viejo = _sha256(RL_DIR_VIEJO / "modelo_ppo.zip") if RL_DIR_VIEJO.exists() else None
print(f"modelo_ppo.zip usado aqui (prueba_rewards/D): {_h_nuevo[:16]}...")
print(f"modelo_ppo.zip de modelo_final (NO usado aqui): {_h_viejo[:16] if _h_viejo else 'no encontrado'}...")
print(f"Son el mismo archivo: {_h_nuevo == _h_viejo}")

Escenario: escalon_1 -- 2 barcos, horas [6, 9], 0.8% poblacion
Semillas de evaluacion: [1001, 1002, 1003, 1004, 1005]
Config de recompensa D (compartida por H0, H3 y PPO): {'tolerancia_incomodidad_min': 0, 'sobrante_normalizador_min': 12, 'penalizacion_maxima_persona': 100000000000.0, 'peso_movimiento': 0, 'premio_por_persona_entregada': 0}
modelo_ppo.zip usado aqui (prueba_rewards/D): 355cda744562b058...
modelo_ppo.zip de modelo_final (NO usado aqui): 80bddd4499bc202c...
Son el mismo archivo: False


In [3]:
model = PPO.load(str(RL_DIR / "modelo_ppo"))
print("Modelo cargado de", RL_DIR / "modelo_ppo.zip")


def construir_entorno_eval():
    return EntornoDemandaAleatoria(
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_recompensa_eval, unidad_demanda=cfg_sim["unidad_demanda"],
    )


_dim_modelo = model.observation_space.shape
_dim_eval = construir_entorno_eval().observation_space.shape
assert _dim_modelo == _dim_eval, (
    f"Desajuste train/eval: el modelo espera observaciones de forma {_dim_modelo}, "
    f"pero {cfg_entren['escalon_base']} da {_dim_eval}."
)
print(f"OK: train y eval usan el mismo escalon, vector aplanado de forma {_dim_eval} en ambos.")

VECNORMALIZE_PATH = RL_DIR / "vecnormalize.pkl"
usar_vecnormalize_eval = VECNORMALIZE_PATH.exists()
print(f"VecNormalize en evaluacion: {usar_vecnormalize_eval}")

Modelo cargado de C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\modelo_rl\prueba_rewards\output\modelos\D\modelo_ppo.zip
OK: train y eval usan el mismo escalon, vector aplanado de forma (46,) en ambos.
VecNormalize en evaluacion: True


## 1. Correr las tres políticas sobre las mismas 5 semillas de evaluación

`correr_agente`/`correr_base` son exactamente el código de `comparacion/notebooks/05` (sin modificar). `correr_h3` es NUEVO -- mismo patrón que `correr_base`, pero con la heurística H3 (con su diccionario de `reservas`, con estado propio, ver `heuristicas/h3_costo_global/src/politica_h3.py`).

In [4]:
def correr_agente(semilla):
    venv_stats = VecNormalize.load(str(VECNORMALIZE_PATH), DummyVecEnv([construir_entorno_eval]))
    venv_stats.training = False
    venv_stats.norm_reward = False

    env = construir_entorno_eval()
    obs, info = env.reset(seed=semilla)
    reward_total = 0.0
    while True:
        obs_norm = venv_stats.normalize_obs(obs)
        accion, _ = model.predict(obs_norm, deterministic=True)
        obs, r, terminated, truncated, info = env.step(accion)
        reward_total += r
        if truncated or terminated:
            break
    return env, reward_total


def correr_base(semilla):
    env = construir_entorno_eval()
    obs, info = env.reset(seed=semilla)
    reward_total = 0.0
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_recompensa_eval)
        accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])
        obs, r, terminated, truncated, info = env.step(accion)
        reward_total += r
        if truncated or terminated:
            break
    return env, reward_total


def correr_h3(semilla):
    env = construir_entorno_eval()
    obs, info = env.reset(seed=semilla)
    reward_total = 0.0
    reservas: dict = {}
    reservas_historial = [dict(reservas)]
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        decisiones, reservas = asignar_flota_h3(libres, estado, matriz_tiempos, env.capacidad_barco, reservas, cfg_recompensa_eval)
        accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])
        obs, r, terminated, truncated, info = env.step(accion)
        reservas_historial.append(dict(reservas))
        reward_total += r
        if truncated or terminated:
            break
    return env, reward_total, reservas_historial


envs_agente, envs_base, envs_h3 = [], [], []
reservas_h3_por_semilla = {}
filas_reward = []
for semilla in SEMILLAS_EVAL:
    env_a, r_a = correr_agente(semilla)
    env_b, r_b = correr_base(semilla)
    env_h, r_h, reservas_hist_h = correr_h3(semilla)
    envs_agente.append(env_a)
    envs_base.append(env_b)
    envs_h3.append(env_h)
    reservas_h3_por_semilla[semilla] = reservas_hist_h
    filas_reward.append({"semilla": semilla, "reward_PPO": r_a, "reward_H0": r_b, "reward_H3": r_h})
    print(f"semilla {semilla}: PPO={r_a:8.2f}  H0={r_b:8.2f}  H3={r_h:8.2f}")

df_reward = pd.DataFrame(filas_reward)
df_reward.to_csv(RES_DIR / "reward_por_semilla.csv", index=False)

semilla 1001: PPO=-7138.96  H0=-1671.69  H3=-1510.40


semilla 1002: PPO=-5274.30  H0=-4527.19  H3=-2448.02


semilla 1003: PPO=-7414.19  H0=-2834.06  H3=-2757.74


semilla 1004: PPO=-16555.58  H0=-15297.35  H3=-7016.17


semilla 1005: PPO=-10244.56  H0=-1687.28  H3=-1632.53


## 2. Combinar las 5 corridas de cada política -- métricas agregadas

In [5]:
corrida_agente = met.combinar_corridas(envs_agente)
corrida_base = met.combinar_corridas(envs_base)
corrida_h3 = met.combinar_corridas(envs_h3)

reporte_agente = met.reporte_completo(corrida_agente)
reporte_base = met.reporte_completo(corrida_base)
reporte_h3 = met.reporte_completo(corrida_h3)

print("Conservacion PPO:", reporte_agente["conservacion"]["cuadra"])
print("Conservacion H0: ", reporte_base["conservacion"]["cuadra"])
print("Conservacion H3: ", reporte_h3["conservacion"]["cuadra"])

Conservacion PPO: True
Conservacion H0:  True
Conservacion H3:  True


## 3. Tabla oficial agregada (media ± std, 5 semillas) -- las tres políticas, mismas métricas que `dia_completo/`

In [6]:
def fila_metricas(env, politica, semilla) -> dict:
    cons = met.verificar_conservacion(env)
    glob = met.metricas_globales(env)
    usr = met.metricas_por_usuario(env)
    por_barco = met.metricas_por_barco(env)
    backlog = met.sin_atender_al_final_por_par(env)
    movs = [e for e in env.log_eventos if e["tipo"] == "movimiento"]
    movimientos_vacios = sum(1 for m in movs if m["ocupacion"] == 0)
    viajes = list(met._tiempos_viaje_por_unidad(env).values())
    return {
        "politica": politica, "semilla": semilla, "conservacion_cuadra": cons["cuadra"],
        "pct_atendidas": glob["pct_atendidas"],
        "backlog_final_total": int(backlog["sin_atender"].sum()) if len(backlog) else 0,
        "espera_media_min": glob["espera_media_min"], "espera_p95_min": usr["espera_min"]["p95"],
        "espera_max_min": usr["espera_min"]["max"],
        "viaje_medio_min": float(np.mean(viajes)) if viajes else float("nan"),
        "sistema_medio_min": glob["sistema_medio_min"],
        "sistema_p95_min": usr["sistema_min"]["p95"], "sistema_max_min": usr["sistema_min"]["max"],
        "movimientos_totales": int(por_barco["movimientos"].sum()),
        "movimientos_vacios": movimientos_vacios, "movimientos_con_carga": len(movs) - movimientos_vacios,
        "ocupacion_pct_capacidad": float(por_barco["ocupacion_media"].mean()) / env.capacidad_barco * 100.0,
        "pct_esperando_flota": float(por_barco["pct_esperando"].mean()),
    }


filas_todas = []
for semilla, env_a, env_b, env_h in zip(SEMILLAS_EVAL, envs_agente, envs_base, envs_h3):
    filas_todas.append(fila_metricas(env_b, "H0", semilla))
    filas_todas.append(fila_metricas(env_h, "H3", semilla))
    filas_todas.append(fila_metricas(env_a, "PPO", semilla))

resultados_por_semilla = pd.DataFrame(filas_todas)
resultados_por_semilla.to_csv(RES_DIR / "resultados_por_semilla_h0_h3_ppo.csv", index=False)
assert resultados_por_semilla["conservacion_cuadra"].all(), "Conservacion NO cuadra en alguna corrida"

METRICAS = [
    "pct_atendidas", "backlog_final_total", "espera_media_min", "espera_p95_min", "espera_max_min",
    "viaje_medio_min", "sistema_medio_min", "sistema_p95_min", "sistema_max_min",
    "movimientos_totales", "movimientos_vacios", "movimientos_con_carga",
    "ocupacion_pct_capacidad", "pct_esperando_flota",
]
filas_tabla = []
for pol in ["H0", "H3", "PPO"]:
    df_pol = resultados_por_semilla[resultados_por_semilla.politica == pol]
    fila = {"politica": pol, "n_semillas": len(df_pol)}
    for m in METRICAS:
        fila[f"{m}_mean"] = df_pol[m].mean()
        fila[f"{m}_std"] = df_pol[m].std()
    filas_tabla.append(fila)

tabla_oficial = pd.DataFrame(filas_tabla).set_index("politica")
tabla_oficial.to_csv(RES_DIR / "tabla_comparativa_h0_h3_ppo_completa.csv")
tabla_oficial.round(3)

,n_semillas,pct_atendidas_mean,pct_atendidas_std,backlog_final_total_mean,backlog_final_total_std,espera_media_min_mean,espera_media_min_std,espera_p95_min_mean,espera_p95_min_std,espera_max_min_mean,...,movimientos_totales_mean,movimientos_totales_std,movimientos_vacios_mean,movimientos_vacios_std,movimientos_con_carga_mean,movimientos_con_carga_std,ocupacion_pct_capacidad_mean,ocupacion_pct_capacidad_std,pct_esperando_flota_mean,pct_esperando_flota_std
politica,,,,,,,,,,,,,,,,,,,,,
H0,5,89.342,4.959,17.2,5.167,14.524,4.641,32.612,16.223,33.370,...,31.6,3.647,16.0,3.082,15.6,1.817,11.795,2.990,32.857,8.409
H3,5,93.403,4.936,10.2,7.050,12.905,2.842,27.887,6.869,29.400,...,28.6,2.966,12.8,1.483,15.8,1.789,12.249,3.233,38.901,10.105
PPO,5,87.170,8.292,20.4,12.198,20.796,3.305,51.798,10.206,55.928,...,37.2,1.304,23.2,1.643,14.0,0.707,11.432,3.293,24.505,2.114


## 4. Gráficas comparativas -- barras con error (PNG listas para la presentación)

In [7]:
def _grafico_barras(metricas: list[str], titulo: str, fname: str, figsize=(12, 4)):
    n = len(metricas)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    politicas = ["H0", "H3", "PPO"]
    for ax, m in zip(axes, metricas):
        medias = [tabla_oficial.loc[p, f"{m}_mean"] for p in politicas]
        stds = [tabla_oficial.loc[p, f"{m}_std"] for p in politicas]
        ax.bar(politicas, medias, yerr=stds, capsize=5, color=[COLOR[p] for p in politicas])
        ax.set_title(m, fontsize=10)
        ax.tick_params(axis="both", labelsize=9)
    fig.suptitle(titulo, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=200, bbox_inches="tight")
    plt.show()
    return fig


_grafico_barras(["pct_atendidas", "backlog_final_total"],
                 f"Servicio -- H0 vs H3 vs PPO (media +/- std, {len(SEMILLAS_EVAL)} semillas, escalon 1)",
                 "barras_servicio.png", figsize=(8, 4))

C:\Users\hfons\AppData\Local\Temp\ipykernel_35916\3441298044.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 800x400 with 2 Axes>

In [8]:
_grafico_barras(["espera_media_min", "espera_p95_min", "espera_max_min", "sistema_medio_min"],
                 f"Tiempos de espera y en sistema -- H0 vs H3 vs PPO (media +/- std, {len(SEMILLAS_EVAL)} semillas, escalon 1)",
                 "barras_tiempos.png", figsize=(15, 4))

C:\Users\hfons\AppData\Local\Temp\ipykernel_35916\3441298044.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 1500x400 with 4 Axes>

In [9]:
_grafico_barras(["movimientos_totales", "movimientos_vacios", "movimientos_con_carga", "ocupacion_pct_capacidad", "pct_esperando_flota"],
                 f"Operación de flota -- H0 vs H3 vs PPO (media +/- std, {len(SEMILLAS_EVAL)} semillas, escalon 1)",
                 "barras_operacion_flota.png", figsize=(18, 4))

C:\Users\hfons\AppData\Local\Temp\ipykernel_35916\3441298044.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 1800x400 with 5 Axes>

In [10]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
politicas = ["H0", "H3", "PPO"]
for ax, m in zip(axes.flat, METRICAS):
    medias = [tabla_oficial.loc[p, f"{m}_mean"] for p in politicas]
    stds = [tabla_oficial.loc[p, f"{m}_std"] for p in politicas]
    ax.bar(politicas, medias, yerr=stds, capsize=4, color=[COLOR[p] for p in politicas])
    ax.set_title(m, fontsize=10)
    ax.tick_params(labelsize=8)
for ax in axes.flat[len(METRICAS):]:
    ax.axis("off")

handles = [mpatches.Patch(color=COLOR[p], label=p) for p in politicas]
fig.legend(handles=handles, loc="upper right", fontsize=11)
fig.suptitle(f"Dashboard comparativo -- H0 vs H3 vs PPO (escalon 1, 2 barcos, "
             f"media +/- std sobre {len(SEMILLAS_EVAL)} semillas)", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIG_DIR / "dashboard_comparativo_h0_h3_ppo.png", dpi=200, bbox_inches="tight")
plt.show()

C:\Users\hfons\AppData\Local\Temp\ipykernel_35916\3991714322.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Elegir la mejor y la peor semilla para el agente (por `sistema_medio_min`, no por reward)

Mismo criterio anti-Goodhart que `comparacion/notebooks/05`.

In [11]:
filas_por_semilla = []
for semilla, env_a in zip(SEMILLAS_EVAL, envs_agente):
    rep_a = met.reporte_completo(env_a)
    filas_por_semilla.append({"semilla": semilla, "sistema_medio_min_agente": rep_a["globales"]["sistema_medio_min"]})

tabla_por_semilla = pd.DataFrame(filas_por_semilla)
semilla_mejor = int(tabla_por_semilla.loc[tabla_por_semilla["sistema_medio_min_agente"].idxmin(), "semilla"])
semilla_peor = int(tabla_por_semilla.loc[tabla_por_semilla["sistema_medio_min_agente"].idxmax(), "semilla"])
SEMILLAS_DETALLE = [semilla_mejor, semilla_peor]
print(f"Mejor semilla para el agente (menor tiempo medio en sistema): {semilla_mejor}")
print(f"Peor semilla para el agente (mayor tiempo medio en sistema):  {semilla_peor}")

Mejor semilla para el agente (menor tiempo medio en sistema): 1003
Peor semilla para el agente (mayor tiempo medio en sistema):  1004


## 6. Gráficas de detalle -- mejor y peor semilla

Perfil de espera y desglose de recompensa se comparan EN UN SOLO gráfico (las 3 políticas overlaid) -- aquí sí es apples-to-apples (misma fórmula de recompensa para las 3, ver introducción). Ocupación de flota y heatmap se dejan por política (overlayarlos con solo 2 barcos por política ya es denso).

In [12]:
def _perfil_espera_comparado(envs_dict: dict, titulo: str):
    fig = go.Figure()
    for pol, env in envs_dict.items():
        minutos = [f["tiempo"]["minuto_del_dia"] for f in env.historial_estados]
        esperando = [sum(v["personas"] for v in f["demanda"].values()) for f in env.historial_estados]
        fig.add_trace(go.Scatter(x=minutos, y=esperando, mode="lines", name=pol, line=dict(color=COLOR[pol], width=2)))
    fig.update_layout(title=titulo, xaxis_title="Minuto del día", yaxis_title="Personas esperando", template="plotly_white")
    return fig


def _desglose_comparado(envs_dict: dict, titulo: str):
    fig = go.Figure()
    for pol, env in envs_dict.items():
        df = met.desglose_recompensa(env)
        fig.add_trace(go.Scatter(x=df["minuto"], y=df["total"], mode="lines", name=pol, line=dict(color=COLOR[pol], width=2)))
    fig.update_layout(title=titulo, xaxis_title="Minuto del día", yaxis_title="Reward total del paso (Reward D, misma formula para las 3)",
                       template="plotly_white")
    return fig


for semilla in SEMILLAS_DETALLE:
    idx = SEMILLAS_EVAL.index(semilla)
    envs_semilla = {"H0": envs_base[idx], "H3": envs_h3[idx], "PPO": envs_agente[idx]}
    etiqueta = "MEJOR" if semilla == semilla_mejor else "PEOR"

    fig = _perfil_espera_comparado(envs_semilla, f"People waiting over time -- H0 vs H3 vs PPO (semilla {semilla}, {etiqueta} para PPO)")
    fig.write_html(FIG_DIR / f"perfil_espera_comparado_semilla{semilla}.html")
    fig.write_image(FIG_DIR / f"perfil_espera_comparado_semilla{semilla}.png", scale=2, width=900, height=500)
    fig.show()

    fig = _desglose_comparado(envs_semilla, f"Reward breakdown (total) -- H0 vs H3 vs PPO (semilla {semilla}, {etiqueta} para PPO)")
    fig.write_html(FIG_DIR / f"desglose_recompensa_comparado_semilla{semilla}.html")
    fig.write_image(FIG_DIR / f"desglose_recompensa_comparado_semilla{semilla}.png", scale=2, width=900, height=500)
    fig.show()

In [13]:
for semilla in SEMILLAS_DETALLE:
    idx = SEMILLAS_EVAL.index(semilla)
    envs_semilla = {"H0": envs_base[idx], "H3": envs_h3[idx], "PPO": envs_agente[idx]}

    fig_heatmaps = make_subplots(rows=1, cols=3, subplot_titles=["H0", "H3", "PPO"])
    for i, (pol, env) in enumerate(envs_semilla.items(), start=1):
        h = viz.graficar_heatmap_cumplimiento(env)
        fig_heatmaps.add_trace(h.data[0], row=1, col=i)
    fig_heatmaps.update_layout(title=f"% atendidas por par O-D -- H0 vs H3 vs PPO (semilla {semilla})",
                                template="plotly_white", height=420, width=1200)
    fig_heatmaps.write_html(FIG_DIR / f"heatmap_cumplimiento_comparado_semilla{semilla}.html")
    fig_heatmaps.write_image(FIG_DIR / f"heatmap_cumplimiento_comparado_semilla{semilla}.png", scale=2, width=1200, height=420)
    fig_heatmaps.show()

    for pol, env in envs_semilla.items():
        fig = viz.graficar_ocupacion_flota(env)
        fig.update_layout(title=f"Fleet occupancy over time -- {pol} (semilla {semilla})")
        fig.write_html(FIG_DIR / f"ocupacion_flota_{pol}_semilla{semilla}.html")
        fig.write_image(FIG_DIR / f"ocupacion_flota_{pol}_semilla{semilla}.png", scale=2, width=900, height=500)

        fig = viz.graficar_sin_atender_al_final(env)
        fig.update_layout(title=f"Backlog at end of run -- {pol} (semilla {semilla})")
        fig.write_html(FIG_DIR / f"backlog_{pol}_semilla{semilla}.html")
        fig.write_image(FIG_DIR / f"backlog_{pol}_semilla{semilla}.png", scale=2, width=900, height=500)

print("Graficas de ocupacion/backlog (HTML + PNG) guardadas en", FIG_DIR)

Graficas de ocupacion/backlog (HTML + PNG) guardadas en C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo\figuras


## 7. Videos (.mp4) -- mejor y peor semilla, las tres políticas

Reemplaza el GIF: un .mp4 sí se puede pausar insertado en PowerPoint. Se genera con `imageio`+`ffmpeg` (bundled), reusando `visualizacion.dibujar_frame` para cada cuadro -- mismo dibujo exacto que usan los GIF del resto del proyecto, solo cambia el formato de salida.

In [14]:
gdf_nodos, gdf_rutas = viz.cargar_capas_mapa(cfg_bb, BB_DIR / "02_ruteo_navegable" / "output" / "rutas_navegables.geojson")
fondo_mapa = viz.descargar_fondo_mapa(gdf_nodos)


def animar_corrida_mp4(historial, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad_barco, out_path, fps=6, fondo=None,
                        figsize=(6.4, 6.4), dpi=100):
    '''Mismo dibujo que viz.animar_corrida (reusa viz.dibujar_frame sin
    modificarlo), pero escribe .mp4 (imageio+ffmpeg) en vez de .gif
    (PillowWriter) -- un video SI se puede pausar insertado en PowerPoint,
    un GIF no de forma confiable. figsize/dpi elegidos multiplos de 16 en
    pixeles (640x640) para evitar que el encoder tenga que rellenar/recortar.
    '''
    if fondo is None:
        fondo = viz.descargar_fondo_mapa(gdf_nodos)
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.94, bottom=0.01)
    writer = imageio.get_writer(str(out_path), fps=fps, codec="libx264", quality=8,
                                 macro_block_size=None, pixelformat="yuv420p")
    try:
        for frame in historial:
            viz.dibujar_frame(ax, frame, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad_barco, fondo=fondo)
            fig.canvas.draw()
            img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            writer.append_data(img)
    finally:
        writer.close()
        plt.close(fig)
    return out_path


def animar_combinado_mp4(historiales: dict, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad_barco, out_path,
                          titulo_extra: str, fps=6, fondo=None, figsize=(19.2, 6.4), dpi=100):
    '''Version combinada 1x3 (H0/H3/PPO lado a lado) como .mp4.'''
    if fondo is None:
        fondo = viz.descargar_fondo_mapa(gdf_nodos)
    politicas = list(historiales.keys())
    n_frames = len(historiales[politicas[0]])
    fig, axes = plt.subplots(1, len(politicas), figsize=figsize, dpi=dpi)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.88, bottom=0.02, wspace=0.05)
    writer = imageio.get_writer(str(out_path), fps=fps, codec="libx264", quality=8,
                                 macro_block_size=None, pixelformat="yuv420p")
    try:
        for i in range(n_frames):
            for ax, pol in zip(axes, politicas):
                frame = historiales[pol][i]
                viz.dibujar_frame(ax, frame, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad_barco, fondo=fondo)
                ax.set_title(pol, fontsize=13, fontweight="bold")
            hh, mm = divmod(int(historiales[politicas[0]][i]["tiempo"]["minuto_del_dia"]), 60)
            fig.suptitle(f"{titulo_extra} -- {hh:02d}:{mm:02d}", fontsize=13)
            fig.canvas.draw()
            img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            writer.append_data(img)
    finally:
        writer.close()
        plt.close(fig)
    return out_path


for semilla in SEMILLAS_DETALLE:
    idx = SEMILLAS_EVAL.index(semilla)
    etiqueta = "mejor" if semilla == semilla_mejor else "peor"
    envs_semilla = {"H0": envs_base[idx], "H3": envs_h3[idx], "PPO": envs_agente[idx]}

    for pol, env in envs_semilla.items():
        out_mp4 = FIG_DIR / f"video_{pol}_semilla{semilla}_{etiqueta}.mp4"
        animar_corrida_mp4(env.historial_estados, gdf_nodos, gdf_rutas, matriz_tiempos, env.capacidad_barco,
                            out_mp4, fps=6, fondo=fondo_mapa)
        print("Guardado:", out_mp4)

    historiales = {pol: env.historial_estados for pol, env in envs_semilla.items()}
    out_mp4_combinado = FIG_DIR / f"video_h0_h3_ppo_comparado_semilla{semilla}_{etiqueta}.mp4"
    animar_combinado_mp4(historiales, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad,
                          out_mp4_combinado, f"H0 vs H3 vs PPO (semilla {semilla}, {etiqueta} para PPO)", fps=6, fondo=fondo_mapa)
    print("Guardado:", out_mp4_combinado)

Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo\figuras\video_H0_semilla1003_mejor.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo\figuras\video_H3_semilla1003_mejor.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo\figuras\video_PPO_semilla1003_mejor.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo\figuras\video_h0_h3_ppo_comparado_semilla1003_mejor.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo\figuras\video_H0_semilla1004_peor.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo\figuras\video_H3_semilla1004_peor.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo\figuras\video_PPO_semilla1004_peor.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\escalon_1_h0_h3_ppo\figuras\video_h0_h3_ppo_comparado_semilla1004_peor.mp4


## 8. Interactive step-by-step player (English, live kernel only)

Play/pause/slider + a side text panel (boat state, decisions, queues, step reward -- and for H3, the active reservations) that updates as you scroll minute by minute. Same widget pattern as `visualizacion.reproductor_interactivo` (reused, not modified), rewritten in English and extended with reservations for H3. **Only works with a live Jupyter kernel** -- open this notebook in VS Code / Jupyter Lab and run these cells yourself; the buttons will not respond in this already-executed, saved copy. One cell per policy/seed (not a shared loop) -- with several `ipywidgets` created in the same loop, only the last one ends up responding reliably.

In [15]:
import io
import ipywidgets as widgets
from IPython.display import Image, clear_output, display


def _state_text_en(env, frame: dict, reservas: dict | None = None) -> str:
    t = frame["tiempo"]["minuto_del_dia"]
    decisions = [e for e in env.log_eventos if e["tipo"] == "decision" and abs(e["minuto"] - t) < 1e-6]
    reward_step = next((r for r in env.log_recompensa if abs(r["minuto"] - t) < 1e-6), None)

    hh, mm = divmod(int(t), 60)
    lines = [f"=== {hh:02d}:{mm:02d} ===", "", "Boats:"]
    for i, b in enumerate(frame["barcos"]):
        dec = next((d["decision"] for d in decisions if d["barco_id"] == f"barco_{i}"), None)
        status = "idle" if b["libre"] else f"{b['origen']}->{b['destino']} ({b['min_para_llegar']:.0f} min left)"
        dec_txt = f", decided: {dec}" if dec is not None else ""
        lines.append(f"  boat_{i}: {status}, occupancy={b['ocupacion']}{dec_txt}")

    lines.append("")
    lines.append("Queues (pairs with people waiting):")
    any_queue = False
    for clave, v in frame["demanda"].items():
        if v["personas"] > 0:
            any_queue = True
            lines.append(f"  {clave}: {v['personas']} people, oldest waiting {v['espera_max']:.1f} min")
    if not any_queue:
        lines.append("  (none)")

    if reservas is not None:
        lines.append("")
        lines.append("H3 active reservations (boat -> queue it is committed to):")
        if not reservas:
            lines.append("  (none)")
        else:
            for bid, (o, d) in reservas.items():
                lines.append(f"  {bid} -> {o}->{d}")

    lines.append("")
    if reward_step:
        lines.append(f"Step reward: {reward_step['total']:.3f}")
        lines.append(f"  discomfort = {reward_step['incomodidad']:.3f}")
        lines.append(f"  movement   = {reward_step['movimiento']:.3f}")

    return "\n".join(lines)


def interactive_player(env, gdf_nodos, gdf_rutas, matriz_tiempos, title: str,
                        reservas_historial: list | None = None, fps: int = 3, fondo=None):
    '''English scrub player (play/pause/slider) with a side panel: boat
    state, decisions, queues and -- for H3, via reservas_historial -- the
    active reservations at that step. Same pattern as
    visualizacion.reproductor_interactivo (continuous_update=False so
    dragging the slider does not flood redraws), reused without modifying
    that file, rewritten in English. Only works with a LIVE Jupyter kernel.
    '''
    historial = env.historial_estados
    n_steps = len(historial)
    if fondo is None:
        fondo = viz.descargar_fondo_mapa(gdf_nodos)

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    plt.close(fig)

    out_map = widgets.Output()
    out_text = widgets.Output(layout=widgets.Layout(width="380px", border="1px solid #ccc", padding="8px"))

    slider = widgets.IntSlider(min=0, max=n_steps - 1, step=1, description="Step", continuous_update=False)
    play = widgets.Play(min=0, max=n_steps - 1, step=1, interval=int(1000 / fps), description="Play")
    widgets.jslink((play, "value"), (slider, "value"))

    back_btn = widgets.Button(description="< Prev step", layout=widgets.Layout(width="140px"))
    fwd_btn = widgets.Button(description="Next step >", layout=widgets.Layout(width="140px"))

    def _update(change):
        i = slider.value
        frame = historial[i]
        reservas = reservas_historial[i] if reservas_historial is not None else None
        viz.dibujar_frame(ax, frame, gdf_nodos, gdf_rutas, matriz_tiempos, env.capacidad_barco, fondo=fondo)
        ax.set_title(title, fontsize=12)
        with out_map:
            clear_output(wait=True)
            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
            display(Image(data=buf.getvalue()))
        with out_text:
            clear_output(wait=True)
            print(_state_text_en(env, frame, reservas))

    def _step_fwd(b):
        slider.value = min(slider.value + 1, n_steps - 1)

    def _step_back(b):
        slider.value = max(slider.value - 1, 0)

    fwd_btn.on_click(_step_fwd)
    back_btn.on_click(_step_back)
    slider.observe(_update, names="value")
    _update(None)

    controls = widgets.HBox([back_btn, fwd_btn, play, slider])
    panel = widgets.HBox([out_map, out_text])
    ui = widgets.VBox([controls, panel])
    display(ui)
    return ui

In [16]:
idx = SEMILLAS_EVAL.index(semilla_mejor)
print(f"=== Player: H0, seed {semilla_mejor} (BEST for PPO) ===")
interactive_player(envs_base[idx], gdf_nodos, gdf_rutas, matriz_tiempos, f"H0 -- seed {semilla_mejor}", fondo=fondo_mapa)

=== Player: H0, seed 1003 (BEST for PPO) ===


In [17]:
idx = SEMILLAS_EVAL.index(semilla_mejor)
print(f"=== Player: H3, seed {semilla_mejor} (BEST for PPO) ===")
interactive_player(envs_h3[idx], gdf_nodos, gdf_rutas, matriz_tiempos, f"H3 -- seed {semilla_mejor}",
                    reservas_historial=reservas_h3_por_semilla[semilla_mejor], fondo=fondo_mapa)

=== Player: H3, seed 1003 (BEST for PPO) ===


In [18]:
idx = SEMILLAS_EVAL.index(semilla_mejor)
print(f"=== Player: PPO, seed {semilla_mejor} (BEST for PPO) ===")
interactive_player(envs_agente[idx], gdf_nodos, gdf_rutas, matriz_tiempos, f"PPO -- seed {semilla_mejor}", fondo=fondo_mapa)

=== Player: PPO, seed 1003 (BEST for PPO) ===


In [19]:
idx = SEMILLAS_EVAL.index(semilla_peor)
print(f"=== Player: H0, seed {semilla_peor} (WORST for PPO) ===")
interactive_player(envs_base[idx], gdf_nodos, gdf_rutas, matriz_tiempos, f"H0 -- seed {semilla_peor}", fondo=fondo_mapa)

=== Player: H0, seed 1004 (WORST for PPO) ===


In [20]:
idx = SEMILLAS_EVAL.index(semilla_peor)
print(f"=== Player: H3, seed {semilla_peor} (WORST for PPO) ===")
interactive_player(envs_h3[idx], gdf_nodos, gdf_rutas, matriz_tiempos, f"H3 -- seed {semilla_peor}",
                    reservas_historial=reservas_h3_por_semilla[semilla_peor], fondo=fondo_mapa)

=== Player: H3, seed 1004 (WORST for PPO) ===


In [21]:
idx = SEMILLAS_EVAL.index(semilla_peor)
print(f"=== Player: PPO, seed {semilla_peor} (WORST for PPO) ===")
interactive_player(envs_agente[idx], gdf_nodos, gdf_rutas, matriz_tiempos, f"PPO -- seed {semilla_peor}", fondo=fondo_mapa)

=== Player: PPO, seed 1004 (WORST for PPO) ===


## 9. Veredicto honesto (automático, no basado en reward)

In [22]:
print("=" * 100)
print(f"COMPARACION FINAL -- H0 vs H3 vs PPO (escalon 1, 2 barcos, semillas {SEMILLAS_EVAL})")
print("=" * 100)
print(tabla_oficial[[f"{m}_mean" for m in METRICAS]].round(2).to_string())

print("\nMejor politica por metrica (automatico -- pct_atendidas: mayor mejor; el resto: menor mejor).")
for m in ["pct_atendidas"]:
    mejor = tabla_oficial[f"{m}_mean"].idxmax()
    print(f"  {m}: mejor = {mejor} ({tabla_oficial.loc[mejor, f'{m}_mean']:.2f})")
for m in ["backlog_final_total", "espera_media_min", "espera_p95_min", "espera_max_min", "sistema_medio_min"]:
    mejor = tabla_oficial[f"{m}_mean"].idxmin()
    print(f"  {m}: mejor (menor) = {mejor} ({tabla_oficial.loc[mejor, f'{m}_mean']:.2f})")

COMPARACION FINAL -- H0 vs H3 vs PPO (escalon 1, 2 barcos, semillas [1001, 1002, 1003, 1004, 1005])
          pct_atendidas_mean  backlog_final_total_mean  espera_media_min_mean  espera_p95_min_mean  espera_max_min_mean  viaje_medio_min_mean  sistema_medio_min_mean  sistema_p95_min_mean  sistema_max_min_mean  movimientos_totales_mean  movimientos_vacios_mean  movimientos_con_carga_mean  ocupacion_pct_capacidad_mean  pct_esperando_flota_mean
politica                                                                                                                                                                                                                                                                                                                                                
H0                     89.34                      17.2                  14.52                32.61                33.37                 10.11                   24.64                 42.02                 43.38 